In [2]:
import pandas as pd

df=pd.read_csv("Customer-Churn.csv")

print(df.head())
print(df.info())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [3]:
print("Class distribution:",df["Churn"].value_counts())

Class distribution: Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [7]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

label=LabelEncoder()
for cols in df.select_dtypes(include='str').columns:
    if cols!='Churn':
        df[cols]=label.fit_transform(df[cols])

df['Churn']=label.fit_transform(df['Churn'])

scaler=StandardScaler()
numerical_features= ['tenure','MonthlyCharges','TotalCharges']
df[numerical_features]=scaler.fit_transform(df[numerical_features])

In [ ]:
from sklearn.model_selection import train_test_split

x=df.drop('Churn',axis=1)
y=df['Churn']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [10]:
from imblearn.over_sampling import SMOTE

smote=SMOTE(random_state=42)
x_train_resampled,y_train_resampled=smote.fit_resample(x_train,y_train)

print("class distribution after resampling:")
print(pd.Series(y_train_resampled).value_counts())

class distribution after resampling:
Churn
0    4138
1    4138
Name: count, dtype: int64


In [11]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

rf_model=RandomForestClassifier(random_state=42)
rf_model.fit(x_train_resampled,y_train_resampled)
rf_pred=rf_model.predict(x_test)

xgb_model=XGBClassifier(eval_metric='logloss',random_state=42)
xgb_model.fit(x_train_resampled,y_train_resampled)
xgb_pred=xgb_model.predict(x_test)

lg_model=LGBMClassifier(random_state=42)
lg_model.fit(x_train_resampled,y_train_resampled)
lg_pred=lg_model.predict(x_test)

In [12]:
from sklearn.metrics import roc_auc_score

roc_rf=roc_auc_score(y_test, rf_model.predict_proba(x_test)[:,1])
roc_xg=roc_auc_score(y_test,xgb_model.predict_proba(x_test)[:,1])
roc_lg=roc_auc_score(y_test,lg_model.predict_proba(x_test)[:,1])

print("ROC Score of Random Forest:",roc_rf)
print("ROC Score of XGBoost:",roc_xg)
print("ROC Score of LightGBM:",roc_lg)

ROC Score of Random Forest: 0.8294818180877161
ROC Score of XGBoost: 0.8236721976668356
ROC Score of LightGBM: 0.8458496796298404
